# Análise dos Benchmarks de Multiplicação Distribuída

Este notebook lê o arquivo `benchmark_results.csv` gerado pelo `Client.py`, compara os tempos de execução serial e distribuída/paralela e calcula o speedup para diferentes tamanhos de matriz.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8")
%matplotlib inline

CSV_PATH = Path("benchmark_results.csv")

## 1. Carregamento dos resultados

Execute primeiro os servidores e depois o cliente. O cliente adiciona uma linha neste CSV a cada tamanho de matriz testado.

In [ ]:
expected_columns = [
    "matrix_size",
    "serial_time_ms",
    "parallel_time_ms",
    "speedup",
    "num_servers",
    "timestamp",
]

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {CSV_PATH.resolve()}")

df = pd.read_csv(CSV_PATH)

missing = set(expected_columns) - set(df.columns)
if missing:
    raise ValueError(f"O CSV não possui as colunas esperadas: {sorted(missing)}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values(["matrix_size", "timestamp"]).reset_index(drop=True)

df

## 2. Média dos resultados por tamanho

Se você rodar o cliente várias vezes, esta tabela consolida a média por tamanho de matriz e quantidade de servidores.

In [ ]:
if df.empty:
    print("O CSV ainda não possui resultados. Execute o Client.py para preencher o benchmark.")
    summary = df.copy()
else:
    summary = (
        df.groupby(["matrix_size", "num_servers"], as_index=False)
        .agg(
            serial_time_ms=("serial_time_ms", "mean"),
            parallel_time_ms=("parallel_time_ms", "mean"),
            runs=("matrix_size", "count"),
        )
        .sort_values("matrix_size")
    )
    summary["speedup"] = summary["serial_time_ms"] / summary["parallel_time_ms"]

summary

## 3. Gráficos comparativos

O primeiro gráfico compara o tempo serial com o tempo distribuído. O segundo mostra o speedup obtido.

In [ ]:
if summary.empty:
    print("Sem dados para plotar.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(summary["matrix_size"], summary["serial_time_ms"], marker="o", label="Serial")
    ax.plot(summary["matrix_size"], summary["parallel_time_ms"], marker="o", label="Distribuído/paralelo")
    ax.set_title("Tempo de execução por tamanho de matriz")
    ax.set_xlabel("Tamanho da matriz (n x n)")
    ax.set_ylabel("Tempo médio (ms)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(summary["matrix_size"].astype(str), summary["speedup"])
    ax.axhline(1, color="black", linewidth=1, linestyle="--")
    ax.set_title("Speedup da execução distribuída")
    ax.set_xlabel("Tamanho da matriz (n x n)")
    ax.set_ylabel("Speedup = tempo serial / tempo distribuído")
    ax.grid(True, axis="y", alpha=0.3)
    plt.show()

## 4. Interpretação

Em matrizes pequenas, o custo de comunicação via socket e serialização pode ser maior que o ganho de paralelismo. Em matrizes maiores, o cálculo tende a dominar o tempo total e o speedup costuma melhorar, desde que os servidores tenham recursos suficientes.

In [ ]:
if summary.empty:
    print("Ainda não há resultados para analisar.")
else:
    for row in summary.itertuples(index=False):
        print(
            f"Matriz {row.matrix_size}x{row.matrix_size} com {row.num_servers} servidores: "
            f"serial={row.serial_time_ms:.2f} ms, "
            f"distribuído={row.parallel_time_ms:.2f} ms, "
            f"speedup={row.speedup:.2f}x ({row.runs} execução/execuções)."
        )

    best = summary.loc[summary["speedup"].idxmax()]
    print(
        "\nMelhor speedup observado: "
        f"{best['speedup']:.2f}x para matriz {int(best['matrix_size'])}x{int(best['matrix_size'])}."
    )